# Data Cleaning — combined_all_data.csv
**Dataset:** 81.554 komentar Instagram & TikTok (BPJS, Coretax, MBG)  
**Tujuan:** Membersihkan data untuk XGBoost dan IndoBERT (3 kelas: keluhan, saran, pujian)  

## Temuan Analisis Awal
| Masalah | Jumlah | Prioritas |
|---|---|---|
| Missing values (kolom text) | 537 (0.66%) | 🔴 Wajib hapus |
| Duplikat username+text | 3.317 (4.07%) | 🔴 Wajib hapus |
| Duplikat text saja | 10.491 (12.9%) | 🟡 Pertahankan jika user berbeda |
| Hanya emoji (no text) | 6.523 (8.05%) | 🔴 Hapus |
| Teks ≤ 2 kata | 16.024 (19.8%) | 🟡 Evaluasi per kasus |
| Dominan non-latin | 1.475 (1.82%) | 🟡 Filter bahasa |
| Timestamp TikTok ISO 8601 | 37.420 | 🟢 Parse ulang saja |
| Teks terpanjang | 2.200 karakter | 🟡 Cek outlier |


## Cell 1 — Setup & Load Data

In [34]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'Arial', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'savefig.dpi': 150,
    'savefig.bbox': 'tight', 'savefig.facecolor': 'white'
})

# ── PATH — sesuaikan ────────────────────────────────────────────────────────
FILE_IN  = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_cleaning\combined_all_data.csv'
FILE_OUT = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_cleaning\data_cleaned.csv'

df = pd.read_csv(FILE_IN)

# Identifikasi platform dari nama file sumber
df['platform'] = df['source_file'].apply(
    lambda x: 'TikTok' if 'tiktok' in str(x).lower() else 'Instagram'
)

print(f'Data dimuat: {len(df):,} baris | {len(df.columns)} kolom')
print(f'Instagram : {(df["platform"]=="Instagram").sum():,}')
print(f'TikTok    : {(df["platform"]=="TikTok").sum():,}')
print(f'\nKolom: {list(df.columns)}')
df.head(3)

Data dimuat: 81,554 baris | 9 kolom
Instagram : 44,134
TikTok    : 37,420

Kolom: ['id', 'timestamp', 'ownerUsername', 'text', 'likesCount', 'postUrl', 'commentUrl', 'source_file', 'platform']


,id,timestamp,ownerUsername,text,likesCount,postUrl,commentUrl,source_file,platform
0,18131249071561283,2026-04-23T10:43:24+00:00,secretariat_irwn_999,"Dipikir dari sisi manapun, 550 miliar ??? Buat...",4524,https://www.instagram.com/reel/DXd-RSckWCU/,https://www.instagram.com/reel/DXd-RSckWCU/?co...,dataset_instagram_20260525_132650.csv,Instagram
1,18049919705732880,2026-04-23T13:30:10+00:00,peterkarundeng,SETIAP DEBUG MELEDAK 🤣 NGAKAK BANGETTT AMPUNNN...,2735,https://www.instagram.com/reel/DXd-RSckWCU/,https://www.instagram.com/reel/DXd-RSckWCU/?co...,dataset_instagram_20260525_132650.csv,Instagram
2,18191321884373494,2026-05-24T00:24:25+00:00,aabahwarung,"Lolololololooolo abang nya yg ga paham, ga nya...",0,https://www.instagram.com/reel/DXd-RSckWCU/,https://www.instagram.com/reel/DXd-RSckWCU/?co...,dataset_instagram_20260525_132650.csv,Instagram


## Cell 2 — Analisis Mendalam Dataset

In [35]:
# ── Cell 2 FIXED — Analisis Mendalam Dataset ─────────────────────────────────

SEP = '=' * 55

def analisis_dataset(df, label=''):
    texts   = df['text'].dropna()
    lengths = texts.str.len()
    words   = texts.str.split().str.len()

    print(SEP)
    print(f'ANALISIS {label}  ({len(df):,} baris)')
    print(SEP)

    print('\n[A] MISSING VALUES:')
    for col in ['text', 'ownerUsername', 'timestamp']:
        n = df[col].isna().sum()
        print(f'  {col:20s}: {n:,} ({n/len(df)*100:.2f}%)')

    print('\n[B] DUPLIKAT:')
    d1 = df['id'].duplicated().sum()
    d2 = df['text'].dropna().duplicated().sum()
    d3 = df.dropna(subset=['text']).duplicated(
             subset=['ownerUsername','text']).sum()
    print(f'  Duplikat ID            : {d1:,}')
    print(f'  Duplikat teks          : {d2:,} ({d2/len(texts)*100:.2f}%)')
    print(f'  Duplikat user+teks     : {d3:,} ({d3/len(texts)*100:.2f}%)')

    print('\n[C] STATISTIK TEKS:')
    print(f'  Panjang karakter : '
          f'min={lengths.min()} | max={lengths.max():,} | '
          f'mean={lengths.mean():.1f} | median={lengths.median():.0f}')
    print(f'  Jumlah kata      : '
          f'min={words.min()} | max={words.max():,} | '
          f'mean={words.mean():.1f} | median={words.median():.0f}')

    print('\n[D] NOISE PATTERNS:')
    import re
    patterns = [
        ('Teks <= 2 kata',
         words <= 2),
        ('Teks <= 5 karakter',
         lengths <= 5),
        ('Hanya emoji',
         texts.str.fullmatch(
             r'[\U00010000-\U0010ffff\u2600-\u27ff\s]+', na=False)),
        ('Hanya angka/simbol',
         texts.str.fullmatch(r'[\d\s\W]+', na=False)),
        ('Mengandung URL',
         texts.str.contains(r'https?://|www\.', na=False)),
        ('Dominan non-latin',
         texts.str.contains(
             r'[^\x00-\x7F\u00C0-\u024F]{5,}', na=False)
         & ~texts.str.contains(r'[a-zA-Z]{3,}', na=False)),
    ]
    for name, mask in patterns:
        n = mask.sum()
        print(f'  {name:28s}: {n:6,} ({n/len(texts)*100:5.2f}%)')

    print('\n[E] TIMESTAMP:')
    ts_ok = pd.to_datetime(
        df['timestamp'], errors='coerce', utc=True).notna().sum()
    print(f'  Timestamp valid: {ts_ok:,}/{len(df):,}')


analisis_dataset(df, 'DATA MENTAH (SEBELUM CLEANING)')

ANALISIS DATA MENTAH (SEBELUM CLEANING)  (81,554 baris)

[A] MISSING VALUES:
  text                : 537 (0.66%)
  ownerUsername       : 0 (0.00%)
  timestamp           : 0 (0.00%)

[B] DUPLIKAT:
  Duplikat ID            : 0
  Duplikat teks          : 9,955 (12.29%)
  Duplikat user+teks     : 3,217 (3.97%)

[C] STATISTIK TEKS:
  Panjang karakter : min=1 | max=2,200 | mean=75.2 | median=44
  Jumlah kata      : min=1 | max=697 | mean=12.5 | median=8

[D] NOISE PATTERNS:
  Teks <= 2 kata              : 16,024 (19.78%)
  Teks <= 5 karakter          :  8,073 ( 9.96%)
  Hanya emoji                 :  6,523 ( 8.05%)
  Hanya angka/simbol          :  7,288 ( 9.00%)
  Mengandung URL              :    128 ( 0.16%)
  Dominan non-latin           :    800 ( 0.99%)

[E] TIMESTAMP:
  Timestamp valid: 44,134/81,554


## Cell 3 — Visualisasi Distribusi Sebelum Cleaning

In [36]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Analisis Dataset Sebelum Cleaning', fontweight='bold', fontsize=12)

texts_all = df['text'].dropna()

# Plot 1: Platform distribution
ax = axes[0,0]
plat_vc = df['platform'].value_counts()
bars = ax.bar(plat_vc.index, plat_vc.values,
              color=['#2471A3','#C0392B'], edgecolor='white')
for bar, val in zip(bars, plat_vc.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
            f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Distribusi Platform', fontweight='bold')
ax.set_ylabel('Jumlah Komentar')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

# Plot 2: Distribusi panjang teks (karakter)
ax = axes[0,1]
lengths = texts_all.str.len()
ax.hist(lengths.clip(upper=300), bins=60, color='#2471A3', edgecolor='white', alpha=0.8)
ax.axvline(lengths.median(), color='#C0392B', linestyle='--', linewidth=1.5,
           label=f'Median: {lengths.median():.0f}')
ax.set_title('Distribusi Panjang Teks (karakter)', fontweight='bold')
ax.set_xlabel('Panjang (dipotong di 300)'); ax.set_ylabel('Frekuensi')
ax.legend(fontsize=9)

# Plot 3: Distribusi jumlah kata
ax = axes[0,2]
words = texts_all.str.split().str.len()
ax.hist(words.clip(upper=50), bins=50, color='#1D9E75', edgecolor='white', alpha=0.8)
ax.axvline(words.median(), color='#C0392B', linestyle='--', linewidth=1.5,
           label=f'Median: {words.median():.0f} kata')
ax.set_title('Distribusi Jumlah Kata', fontweight='bold')
ax.set_xlabel('Jumlah kata (dipotong di 50)'); ax.set_ylabel('Frekuensi')
ax.legend(fontsize=9)

# Plot 4: Noise patterns
ax = axes[1,0]
noise_labels = ['Missing\ntext','Dup\nuser+teks','Hanya\nemoji','≤2 kata','Non-latin\ndominan','URL']
noise_counts = [
    df['text'].isna().sum(),
    df.dropna(subset=['text']).duplicated(subset=['ownerUsername','text']).sum(),
    texts_all.str.fullmatch(r'[\U00010000-\U0010ffff\u2600-\u27ff\s]+', na=False).sum(),
    (texts_all.str.split().str.len() <= 2).sum(),
    (texts_all.str.contains(r'[^\x00-\x7F\u00C0-\u024F]{5,}', na=False) &
     ~texts_all.str.contains(r'[a-zA-Z]{3,}', na=False)).sum(),
    texts_all.str.contains(r'https?://|www\.', na=False).sum(),
]
colors_n = ['#E74C3C','#E74C3C','#E67E22','#F39C12','#95A5A6','#95A5A6']
bars = ax.barh(noise_labels, noise_counts, color=colors_n, edgecolor='white')
for bar, val in zip(bars, noise_counts):
    ax.text(bar.get_width()+50, bar.get_y()+bar.get_height()/2,
            f'{val:,}', va='center', fontsize=8.5, fontweight='bold')
ax.set_title('Distribusi Noise Pattern', fontweight='bold')
ax.set_xlabel('Jumlah')

# Plot 5: Distribusi likesCount
ax = axes[1,1]
likes = df['likesCount'].clip(upper=100)
ax.hist(likes, bins=50, color='#8E44AD', edgecolor='white', alpha=0.8)
ax.set_title('Distribusi likesCount (clip 100)', fontweight='bold')
ax.set_xlabel('Likes'); ax.set_ylabel('Frekuensi')
zero_likes = (df['likesCount'] == 0).sum()
ax.text(0.6, 0.85, f'{zero_likes:,} komentar\ntanpa likes',
        transform=ax.transAxes, fontsize=9, color='#8E44AD')

# Plot 6: Panjang teks per platform
ax = axes[1,2]
for plat, color in [('Instagram','#2471A3'),('TikTok','#C0392B')]:
    lens = df[df['platform']==plat]['text'].dropna().str.len()
    ax.hist(lens.clip(upper=300), bins=40, alpha=0.6,
            color=color, label=plat, edgecolor='white')
ax.set_title('Panjang Teks per Platform', fontweight='bold')
ax.set_xlabel('Panjang karakter (clip 300)')
ax.set_ylabel('Frekuensi')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('analisis_sebelum_cleaning.png')
plt.show()
print('✅ Gambar tersimpan: analisis_sebelum_cleaning.png')

✅ Gambar tersimpan: analisis_sebelum_cleaning.png


## Cell 4 — Pipeline Data Cleaning (7 Tahap)

Setiap tahap didokumentasikan dengan jumlah data yang berkurang.

| Tahap | Aksi | Alasan |
|---|---|---|
| C1 | Hapus missing values (text null) | Tidak bisa diproses |
| C2 | Parse & normalisasi timestamp | TikTok ISO 8601 format berbeda |
| C3 | Hapus duplikat (username + text) | Menghindari bias TF-IDF |
| C4 | Hapus teks hanya emoji/simbol | Tidak bermakna semantik |
| C5 | Hapus teks ≤ 2 kata | Terlalu pendek untuk klasifikasi |
| C6 | Hapus teks dominan non-latin | Bukan Bahasa Indonesia |
| C7 | Hapus URL standalone | Tidak informatif |


In [37]:
# ══════════════════════════════════════════════════════════════
# CELL 4 FIX — Pipeline Data Cleaning (7 Tahap)
# Perbaikan:
#   C2: timestamp TikTok tidak dihapus, cukup di-parse ulang
#   C4: regex emoji diperbaiki agar tidak over-filter
# ══════════════════════════════════════════════════════════════

import re
import pandas as pd

report    = []
df_clean  = df.copy()

def log_step(df_before, df_after, step, reason):
    removed = len(df_before) - len(df_after)
    report.append({
        'Tahap'  : step,
        'Alasan' : reason,
        'Sebelum': len(df_before),
        'Sesudah': len(df_after),
        'Dihapus': removed,
        'Persen' : f'{removed/len(df_before)*100:.2f}%'
    })
    print(f'[{step}] {reason}')
    print(f'       {len(df_before):,} → {len(df_after):,} '
          f'| dihapus: {removed:,} ({removed/len(df_before)*100:.2f}%)')

print(f'Data awal: {len(df_clean):,} baris\n')

# ── C1: Hapus missing values & teks kosong ──────────────────────────────────
df_prev  = df_clean.copy()
df_clean = df_clean.dropna(subset=['text'])
df_clean = df_clean[df_clean['text'].str.strip() != '']
log_step(df_prev, df_clean, 'C1', 'Hapus missing values & teks kosong')

# ── C1b: Normalisasi newline di dalam teks ───────────────────────────────────
# Masalah: kolom text mengandung \n (newline) di dalam satu komentar
# Contoh:
#   "*Solusi MBG*\nMohon BGN bekerjasama dgn Peruri,\n..."
# Solusi: ganti \n dan \r dengan spasi agar jadi satu baris teks murni

df_prev  = df_clean.copy()

# Ganti newline (\n, \r\n, \r) dengan spasi
df_clean['text'] = df_clean['text'].str.replace(r'\r\n|\r|\n', ' ', regex=True)

# Ganti spasi ganda yang muncul akibat penggantian
df_clean['text'] = df_clean['text'].str.replace(r' {2,}', ' ', regex=True)

# Trim spasi di awal dan akhir
df_clean['text'] = df_clean['text'].str.strip()

# Cek hasilnya
multiline_before = df_prev['text'].str.contains(r'\n', na=False).sum()
multiline_after  = df_clean['text'].str.contains(r'\n', na=False).sum()

print(f'[C1b] Normalisasi newline dalam teks')
print(f'       Teks dengan newline sebelum : {multiline_before:,}')
print(f'       Teks dengan newline sesudah : {multiline_after:,}')
print(f'       Jumlah baris tetap          : {len(df_clean):,}')

# Contoh hasil normalisasi
sample = df_prev[df_prev['text'].str.contains(r'\n', na=False)].head(2)
for _, row in sample.iterrows():
    original = str(row['text'])[:120].replace('\n', '↵')
    cleaned  = df_clean.loc[row.name, 'text'][:120]
    print(f'\n  Sebelum: {original}')
    print(f'  Sesudah: {cleaned}')

# ── C2: Parse timestamp — handle dua format berbeda ─────────────────────────
df_prev = df_clean.copy()

def parse_timestamp(ts):
    """
    Handle dua format:
    Instagram : 2026-04-23T10:43:24+00:00
    TikTok    : 2026-02-25T20:51:09.000Z
    """
    try:
        ts_str = str(ts).strip()
        # Ganti .000Z → +00:00 agar pandas bisa parse keduanya
        ts_str = ts_str.replace('.000Z', '+00:00').replace('Z', '+00:00')
        return pd.Timestamp(ts_str)
    except Exception:
        return pd.NaT

df_clean['timestamp_parsed'] = df_clean['timestamp'].apply(parse_timestamp)

ts_ok   = df_clean['timestamp_parsed'].notna().sum()
ts_fail = df_clean['timestamp_parsed'].isna().sum()

print(f'[C2] Parse timestamp (Instagram + TikTok)')
print(f'     Berhasil di-parse : {ts_ok:,}')
print(f'     Gagal di-parse    : {ts_fail:,}')
print(f'     Total baris tetap : {len(df_clean):,}')

# Verifikasi sample
print(f'\n     Contoh hasil parse:')
ig_sample = df_clean[df_clean['platform']=='Instagram']['timestamp_parsed'].dropna().head(2)
tt_sample = df_clean[df_clean['platform']=='TikTok']['timestamp_parsed'].dropna().head(2)
for t in ig_sample: print(f'     Instagram: {t}')
for t in tt_sample: print(f'     TikTok   : {t}')

log_step(df_prev, df_clean, 'C2', 
         f'Parse timestamp UTC — Instagram (+00:00) & TikTok (.000Z) — {ts_fail} gagal')

# ── C3: Hapus duplikat username + text ──────────────────────────────────────
df_prev  = df_clean.copy()
df_clean = df_clean.drop_duplicates(subset=['ownerUsername', 'text'], keep='first')
log_step(df_prev, df_clean, 'C3', 'Hapus duplikat (username + text identik)')

# ── C4: Hapus teks hanya emoji/simbol — REGEX DIPERBAIKI ────────────────────
# Bug sebelumnya: regex terlalu agresif, menghapus teks bahasa Indonesia
# Fix: cek apakah teks mengandung minimal 2 huruf latin atau angka
df_prev = df_clean.copy()

def has_meaningful_content(text):
    """
    Kembalikan True jika teks mengandung konten bermakna:
    minimal 2 huruf alfabet (a-z, A-Z) atau angka
    Hapus hanya jika benar-benar tidak ada huruf sama sekali
    """
    t = str(text)
    # Hitung huruf alfabet (latin) dalam teks
    latin_chars = re.findall(r'[a-zA-Z]', t)
    # Pertahankan jika ada minimal 2 huruf latin
    return len(latin_chars) >= 2

mask_meaningful = df_clean['text'].apply(has_meaningful_content)
df_clean = df_clean[mask_meaningful]
log_step(df_prev, df_clean, 'C4',
         'Hapus teks tanpa huruf latin (hanya emoji/angka/simbol)')

# ── C5: Hapus teks < 3 kata ──────────────────────────────────────────────────
df_prev = df_clean.copy()

def count_words(text):
    # Hapus mention, URL, hashtag dulu sebelum hitung kata
    t = re.sub(r'@\w+|https?://\S+|#\w+', '', str(text))
    return len(t.split())

df_clean['_wc'] = df_clean['text'].apply(count_words)
df_clean = df_clean[df_clean['_wc'] >= 3]
df_clean = df_clean.drop(columns=['_wc'])
log_step(df_prev, df_clean, 'C5', 'Hapus teks < 3 kata bersih')

# ── C6: Hapus teks dominan non-latin ────────────────────────────────────────
df_prev = df_clean.copy()

def is_indonesian_enough(text):
    t = str(text)
    # Hitung karakter latin (termasuk huruf beraksara latin)
    latin = len(re.findall(r'[a-zA-Z\u00C0-\u024F]', t))
    total = len(re.sub(r'\s', '', t))
    if total == 0:
        return False
    return (latin / total) >= 0.25  # minimal 25% karakter latin

df_clean = df_clean[df_clean['text'].apply(is_indonesian_enough)]
log_step(df_prev, df_clean, 'C6',
         'Hapus teks dominan non-latin (bukan Bahasa Indonesia)')

# ── C7: Hapus teks yang isinya hanya URL ────────────────────────────────────
df_prev   = df_clean.copy()
only_url  = df_clean['text'].str.strip().str.fullmatch(
    r'(https?://\S+\s*)+', na=False
)
df_clean  = df_clean[~only_url]
log_step(df_prev, df_clean, 'C7', 'Hapus teks yang isinya hanya URL')


# ── C8: Bersihkan token [Sticker] dan hapus jika kosong setelahnya ────────────
df_prev  = df_clean.copy()

# Hapus token [Sticker], [sticker], [foto] dari teks
df_clean['text'] = df_clean['text'].str.replace(
    r'\[(?:Sticker|sticker|foto|Foto)\]\s*', '', regex=True
)

# Hapus baris yang menjadi kosong / < 3 kata setelah pembersihan
df_clean['text'] = df_clean['text'].str.strip()
df_clean = df_clean[df_clean['text'] != '']
df_clean['_wc2'] = df_clean['text'].str.split().str.len()
df_clean = df_clean[df_clean['_wc2'] >= 3]
df_clean = df_clean.drop(columns=['_wc2'])

log_step(df_prev, df_clean, 'C8',
         'Hapus token [Sticker]/[foto] — buang baris yang jadi kosong')

# ── Ringkasan ────────────────────────────────────────────────────────────────
print()
total_hapus = len(df) - len(df_clean)
print('=' * 55)
print('RINGKASAN CLEANING')
print('=' * 55)
print(f'  Data awal    : {len(df):,}')
print(f'  Data bersih  : {len(df_clean):,}')
print(f'  Total dihapus: {total_hapus:,} ({total_hapus/len(df)*100:.2f}%)')
print()
print('Distribusi platform setelah cleaning:')
for plat, cnt in df_clean['platform'].value_counts().items():
    pct = cnt / len(df_clean) * 100
    bar = '█' * int(pct / 2)
    print(f'  {plat:12s}: {cnt:,} ({pct:.1f}%) {bar}')

Data awal: 81,554 baris

[C1] Hapus missing values & teks kosong
       81,554 → 81,017 | dihapus: 537 (0.66%)
[C1b] Normalisasi newline dalam teks
       Teks dengan newline sebelum : 2,884
       Teks dengan newline sesudah : 0
       Jumlah baris tetap          : 81,017

  Sebelum: Gimana Rupiah nggak terus melemah coba ...↵Program Pemerintah dengan Anggaran segitu besar, di ambil cuma beberapa % doa
  Sesudah: Gimana Rupiah nggak terus melemah coba ... Program Pemerintah dengan Anggaran segitu besar, di ambil cuma beberapa % doa

  Sebelum: @prabowo @presidenrepublikindonesia @gerindra @fraksipartaigerindra gimana nh ?↵@official.kpk @kejaksaan.ri  boleh di ce
  Sesudah: @prabowo @presidenrepublikindonesia @gerindra @fraksipartaigerindra gimana nh ? @official.kpk @kejaksaan.ri boleh di cek
[C2] Parse timestamp (Instagram + TikTok)
     Berhasil di-parse : 81,017
     Gagal di-parse    : 0
     Total baris tetap : 81,017

     Contoh hasil parse:
     Instagram: 2026-04-23 10:43:24+0

## Cell 5 — Laporan Ringkasan Cleaning

In [38]:
df_report = pd.DataFrame(report)
print('LAPORAN DATA CLEANING')
print('='*70)
print(df_report.to_string(index=False))
print('='*70)

# Platform setelah cleaning
print(f'\nDistribusi platform SESUDAH cleaning:')
for plat, cnt in df_clean['platform'].value_counts().items():
    pct = cnt/len(df_clean)*100
    print(f'  {plat:12s}: {cnt:,} ({pct:.1f}%)')

LAPORAN DATA CLEANING
Tahap                                                              Alasan  Sebelum  Sesudah  Dihapus Persen
   C1                                  Hapus missing values & teks kosong    81554    81017      537  0.66%
   C2 Parse timestamp UTC — Instagram (+00:00) & TikTok (.000Z) — 0 gagal    81017    81017        0  0.00%
   C3                            Hapus duplikat (username + text identik)    81017    77794     3223  3.98%
   C4             Hapus teks tanpa huruf latin (hanya emoji/angka/simbol)    77794    72258     5536  7.12%
   C5                                          Hapus teks < 3 kata bersih    72258    63615     8643 11.96%
   C6               Hapus teks dominan non-latin (bukan Bahasa Indonesia)    63615    63604       11  0.02%
   C7                                    Hapus teks yang isinya hanya URL    63604    63604        0  0.00%
   C8         Hapus token [Sticker]/[foto] — buang baris yang jadi kosong    63604    63539       65  0.10%

Distr

## Cell 6 — Visualisasi Before vs After Cleaning

In [39]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Perbandingan Dataset Sebelum dan Sesudah Cleaning', fontweight='bold', fontsize=12)

# Plot 1: Before vs After total
ax = axes[0,0]
stages = [r['Sesudah'] for r in report]
labels_s = [r['Tahap'] for r in report]
stages.insert(0, len(df))
labels_s.insert(0, 'Awal')
colors_s = ['#2471A3'] + ['#E74C3C' if i < len(stages)-2 else '#1D9E75'
                          for i in range(len(stages)-1)]
ax.bar(labels_s, stages, color=colors_s, edgecolor='white')
for i, (lbl, val) in enumerate(zip(labels_s, stages)):
    ax.text(i, val+200, f'{val:,}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax.set_title('Jumlah Data per Tahap Cleaning', fontweight='bold')
ax.set_ylabel('Jumlah Baris')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.tick_params(axis='x', rotation=30)

# Plot 2: Jumlah yang dihapus per tahap
ax = axes[0,1]
removed = [r['Dihapus'] for r in report]
tahap   = [r['Tahap']   for r in report]
bars = ax.bar(tahap, removed,
              color=['#E74C3C','#3498DB','#E67E22','#F39C12','#9B59B6','#1ABC9C','#95A5A6'],
              edgecolor='white')
for bar, val in zip(bars, removed):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
            f'{val:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_title('Jumlah Data Dihapus per Tahap', fontweight='bold')
ax.set_ylabel('Jumlah Dihapus')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.tick_params(axis='x', rotation=30)

# Plot 3: Distribusi panjang teks BEFORE vs AFTER
ax = axes[1,0]
len_before = df['text'].dropna().str.len().clip(upper=300)
len_after  = df_clean['text'].str.len().clip(upper=300)
ax.hist(len_before, bins=50, alpha=0.5, color='#E74C3C', label=f'Sebelum ({len(df):,})')
ax.hist(len_after,  bins=50, alpha=0.5, color='#1D9E75', label=f'Sesudah ({len(df_clean):,})')
ax.set_title('Distribusi Panjang Teks Before vs After', fontweight='bold')
ax.set_xlabel('Panjang karakter (clip 300)')
ax.set_ylabel('Frekuensi')
ax.legend(fontsize=9)

# Plot 4: Distribusi platform Before vs After
ax = axes[1,1]
x = np.arange(2)
w = 0.35
plat_before = df['platform'].value_counts()
plat_after  = df_clean['platform'].value_counts()
b1 = ax.bar(x-w/2, [plat_before.get('Instagram',0), plat_before.get('TikTok',0)],
            w, label='Sebelum', color='#E74C3C', edgecolor='white', alpha=0.8)
b2 = ax.bar(x+w/2, [plat_after.get('Instagram',0),  plat_after.get('TikTok',0)],
            w, label='Sesudah', color='#1D9E75', edgecolor='white', alpha=0.8)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax.set_title('Platform Before vs After Cleaning', fontweight='bold')
ax.set_ylabel('Jumlah')
ax.set_xticks(x); ax.set_xticklabels(['Instagram','TikTok'])
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('perbandingan_cleaning.png')
plt.show()
print('✅ Gambar tersimpan: perbandingan_cleaning.png')

✅ Gambar tersimpan: perbandingan_cleaning.png


## Cell 7 — Validasi Akhir & Simpan Data Bersih

In [42]:
print('VALIDASI AKHIR DATA BERSIH')
print('='*55)

texts_clean = df_clean['text']

# Cek semua noise sudah bersih
checks = {
    'Missing values'     : texts_clean.isna().sum(),
    'Teks kosong'        : (texts_clean.str.strip() == '').sum(),
    'Duplikat user+text' : df_clean.duplicated(subset=['ownerUsername','text']).sum(),
    'Hanya emoji'        : texts_clean.apply(
                               lambda t: len(re.sub(
                                   r'[\U00010000-\U0010ffff\u2600-\u27ff\s\W]','',str(t)))<2
                           ).sum(),
    'Teks < 3 kata'      : (texts_clean.str.split().str.len() < 3).sum(),
}

all_pass = True
for chk, val in checks.items():
    status = '✅ BERSIH' if val == 0 else f'⚠️  {val:,} tersisa'
    print(f'  {chk:25s}: {status}')
    if val > 0: all_pass = False

print()
print(f'Statistik teks BERSIH:')
lens  = texts_clean.str.len()
words = texts_clean.str.split().str.len()
print(f'  Panjang karakter: mean={lens.mean():.1f} | median={lens.median():.0f} | max={lens.max():,}')
print(f'  Jumlah kata     : mean={words.mean():.1f} | median={words.median():.0f} | max={words.max():,}')

# Kolom final untuk disimpan
cols_save = ['id','timestamp_parsed','ownerUsername','text',
             'likesCount','postUrl','commentUrl','platform','source_file']
df_save = df_clean[cols_save].rename(columns={'timestamp_parsed':'timestamp'})
df_save = df_save.reset_index(drop=True)

# Tambahkan di Cell 7, tepat sebelum baris df_save.to_csv(...)
# Hapus 1 baris emoji yang tersisa
df_clean = df_clean[df_clean['text'].apply(
    lambda t: len(__import__('re').findall(r'[a-zA-Z]', str(t))) >= 2
)]
print(f'Setelah hapus sisa emoji: {len(df_clean):,} baris')

df_save.to_csv(FILE_OUT, index=False, encoding='utf-8-sig')

print(f'\n✅ Data bersih tersimpan: {FILE_OUT}')
print(f'   Total baris : {len(df_save):,}')
print(f'\nLanjutkan ke notebook Labeling_Otomatis_IndoBERT.ipynb!')

VALIDASI AKHIR DATA BERSIH
  Missing values           : ✅ BERSIH
  Teks kosong              : ✅ BERSIH
  Duplikat user+text       : ✅ BERSIH
  Hanya emoji              : ✅ BERSIH
  Teks < 3 kata            : ✅ BERSIH

Statistik teks BERSIH:
  Panjang karakter: mean=90.2 | median=58 | max=2,200
  Jumlah kata     : mean=15.0 | median=10 | max=351
Setelah hapus sisa emoji: 63,538 baris

✅ Data bersih tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_cleaning\data_cleaned.csv
   Total baris : 63,538

Lanjutkan ke notebook Labeling_Otomatis_IndoBERT.ipynb!


## Cell 8 — Ringkasan untuk Skripsi (Bab 3 Sub-bab Data Cleaning)

In [41]:
total_awal   = len(df)
total_bersih = len(df_clean)
total_hapus  = total_awal - total_bersih

print('='*60)
print('RINGKASAN DATA CLEANING UNTUK SKRIPSI')
print('='*60)
print(f'\nData mentah (setelah scraping) : {total_awal:,} baris')
print(f'Data bersih (setelah cleaning) : {total_bersih:,} baris')
print(f'Total data dihapus             : {total_hapus:,} baris ({total_hapus/total_awal*100:.2f}%)')

print(f'\nRincian tahapan cleaning:')
for r in report:
    print(f"  {r['Tahap']}: {r['Alasan']} → -{r['Dihapus']:,} baris")

print(f'\n--- Kalimat siap pakai untuk Bab 3 ---')
print(f'Tahap data cleaning dilakukan terhadap {total_awal:,} baris data mentah')
print(f'hasil scraping dari platform Instagram dan TikTok. Proses cleaning')
print(f'terdiri dari {len(report)} tahap, yaitu: penghapusan missing values ({report[0]["Dihapus"]:,} baris),')
print(f'normalisasi timestamp TikTok format ISO 8601, penghapusan duplikat')
print(f'berdasarkan kombinasi username dan teks ({report[2]["Dihapus"]:,} baris), penghapusan')
print(f'teks yang hanya mengandung emoji ({report[3]["Dihapus"]:,} baris), teks kurang dari 3 kata')
print(f'({report[4]["Dihapus"]:,} baris), teks dominan non-latin ({report[5]["Dihapus"]:,} baris),')
print(f'dan teks yang hanya berisi URL ({report[6]["Dihapus"]:,} baris). Hasil akhir')
print(f'proses cleaning menghasilkan {total_bersih:,} baris data bersih yang siap')
print(f'digunakan untuk proses pelabelan dan pemodelan.')

RINGKASAN DATA CLEANING UNTUK SKRIPSI

Data mentah (setelah scraping) : 81,554 baris
Data bersih (setelah cleaning) : 63,538 baris
Total data dihapus             : 18,016 baris (22.09%)

Rincian tahapan cleaning:
  C1: Hapus missing values & teks kosong → -537 baris
  C2: Parse timestamp UTC — Instagram (+00:00) & TikTok (.000Z) — 0 gagal → -0 baris
  C3: Hapus duplikat (username + text identik) → -3,223 baris
  C4: Hapus teks tanpa huruf latin (hanya emoji/angka/simbol) → -5,536 baris
  C5: Hapus teks < 3 kata bersih → -8,643 baris
  C6: Hapus teks dominan non-latin (bukan Bahasa Indonesia) → -11 baris
  C7: Hapus teks yang isinya hanya URL → -0 baris
  C8: Hapus token [Sticker]/[foto] — buang baris yang jadi kosong → -65 baris

--- Kalimat siap pakai untuk Bab 3 ---
Tahap data cleaning dilakukan terhadap 81,554 baris data mentah
hasil scraping dari platform Instagram dan TikTok. Proses cleaning
terdiri dari 8 tahap, yaitu: penghapusan missing values (537 baris),
normalisasi timestamp